In [15]:

!pip install --upgrade pip


!pip install pandas
!pip install numpy


!pip install scikit-learn

!pip install xgboost


!pip install imbalanced-learn

!pip install matplotlib
!pip install seaborn

Defaulting to user installation because normal site-packages is not writeable
  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Program Files\Python313\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable


...


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
# Library
import pandas as pd
import numpy as np
from collections import Counter

# Preprocessing & split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Metrics
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
# Model
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

# Imbalanced learning (undersampling)
from imblearn.under_sampling import RandomUnderSampler, ClusterCentroids, TomekLinks, EditedNearestNeighbours

# Misc
import warnings
warnings.filterwarnings('ignore')


In [17]:
# Load CSV
df = pd.read_csv("diabetes_dataset.csv")

# Cek 5 baris pertama
print(df.head())

   year  gender   age location  race:AfricanAmerican  race:Asian  \
0  2020  Female  32.0  Alabama                     0           0   
1  2015  Female  29.0  Alabama                     0           1   
2  2015    Male  18.0  Alabama                     0           0   
3  2015    Male  41.0  Alabama                     0           0   
4  2016  Female  52.0  Alabama                     1           0   

   race:Caucasian  race:Hispanic  race:Other  hypertension  heart_disease  \
0               0              0           1             0              0   
1               0              0           0             0              0   
2               0              0           1             0              0   
3               1              0           0             0              0   
4               0              0           0             0              0   

  smoking_history    bmi  hbA1c_level  blood_glucose_level  diabetes  
0           never  27.32          5.0                  10

In [18]:
df = df.drop_duplicates()


In [19]:
df = df.drop(columns=["year", "location"])


In [20]:
df = pd.get_dummies(df, columns=["gender", "smoking_history"], drop_first=True)


In [21]:
X = df.drop("diabetes", axis=1)
y = df["diabetes"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [22]:
# 5️⃣ Random Forest
# ===============================
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'  # penting supaya minoritas terbaca
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\n=== Random Forest (Data Asli) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))

# ===============================
# 6️⃣ XGBoost
# ===============================
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print("\n=== XGBoost (Data Asli) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))


=== Random Forest (Data Asli) ===
Accuracy: 0.970997099709971
F1 Score: 0.7997237569060773
Confusion Matrix:
 [[18260    38]
 [  542  1158]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.98     18298
           1       0.97      0.68      0.80      1700

    accuracy                           0.97     19998
   macro avg       0.97      0.84      0.89     19998
weighted avg       0.97      0.97      0.97     19998


=== XGBoost (Data Asli) ===
Accuracy: 0.96994699469947
F1 Score: 0.7968908415005069
Confusion Matrix:
 [[18218    80]
 [  521  1179]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.98     18298
           1       0.94      0.69      0.80      1700

    accuracy                           0.97     19998
   macro avg       0.95      0.84      0.89     19998
weighted avg       0.97      0.97      0.97     19998



In [23]:
# ===== SMOTE =====
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

print("Distribusi setelah SMOTE:", dict(zip(*np.unique(y_train_sm, return_counts=True))))

# ===== Random Forest dengan SMOTE =====
rf_sm = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)
rf_sm.fit(X_train_sm, y_train_sm)
y_pred_rf_sm = rf_sm.predict(X_test)

print("\n=== Random Forest (SMOTE) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf_sm))
print("F1 Class 1:", f1_score(y_test, y_pred_rf_sm, pos_label=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf_sm))
print("Classification Report:\n", classification_report(y_test, y_pred_rf_sm))

# ===== XGBoost dengan SMOTE =====
xgb_sm = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb_sm.fit(X_train_sm, y_train_sm)
y_pred_xgb_sm = xgb_sm.predict(X_test)

print("\n=== XGBoost (SMOTE) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb_sm))
print("F1 Class 1:", f1_score(y_test, y_pred_xgb_sm, pos_label=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb_sm))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb_sm))


Distribusi setelah SMOTE: {np.int64(0): np.int64(73188), np.int64(1): np.int64(73188)}

=== Random Forest (SMOTE) ===
Accuracy: 0.9667966796679668
F1 Class 1: 0.78314826910516
Confusion Matrix:
 [[18135   163]
 [  501  1199]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.99      0.98     18298
           1       0.88      0.71      0.78      1700

    accuracy                           0.97     19998
   macro avg       0.93      0.85      0.88     19998
weighted avg       0.97      0.97      0.97     19998


=== XGBoost (SMOTE) ===
Accuracy: 0.9693469346934693
F1 Class 1: 0.7943643072794364
Confusion Matrix:
 [[18201    97]
 [  516  1184]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.99      0.98     18298
           1       0.92      0.70      0.79      1700

    accuracy                           0.97     19998
   macro avg       0.95      0.85      0.89   

In [24]:
 #1️⃣ Random Over Sampling (ROS)
# ===============================
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X_train, y_train)
print("Distribusi setelah ROS:", Counter(y_res))

# ===============================
# 2️⃣ Random Forest (ROS)
# ===============================
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)
rf.fit(X_res, y_res)
y_pred_rf = rf.predict(X_test)

print("\n=== Random Forest (ROS) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))

# ===============================
# 3️⃣ XGBoost (ROS)
# ===============================
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb.fit(X_res, y_res)
y_pred_xgb = xgb.predict(X_test)

print("\n=== XGBoost (ROS) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))

Distribusi setelah ROS: Counter({0: 73188, 1: 73188})

=== Random Forest (ROS) ===
Accuracy: 0.968046804680468
F1 Score: 0.791924454575057
Confusion Matrix:
 [[18143   155]
 [  484  1216]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.99      0.98     18298
           1       0.89      0.72      0.79      1700

    accuracy                           0.97     19998
   macro avg       0.93      0.85      0.89     19998
weighted avg       0.97      0.97      0.97     19998


=== XGBoost (ROS) ===
Accuracy: 0.9014401440144014
F1 Score: 0.6129982328686432
Confusion Matrix:
 [[16466  1832]
 [  139  1561]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.90      0.94     18298
           1       0.46      0.92      0.61      1700

    accuracy                           0.90     19998
   macro avg       0.73      0.91      0.78     19998
weighted avg       0.95      0.90

In [25]:
# 1️⃣ SMOTEENN
# ===============================
from imblearn.combine import SMOTEENN
smote_enn = SMOTEENN(random_state=42)
X_res, y_res = smote_enn.fit_resample(X_train, y_train)
print("Distribusi setelah SMOTEENN:", Counter(y_res))


# 2️⃣ Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)
rf.fit(X_res, y_res)
y_pred_rf = rf.predict(X_test)

print("\n=== Random Forest  ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf, pos_label=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))

# 3️⃣ XGBoost
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb.fit(X_res, y_res)
y_pred_xgb = xgb.predict(X_test)

print("\n=== XGBoost ===")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb, pos_label=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))


Distribusi setelah SMOTEENN: Counter({1: 72277, 0: 63393})

=== Random Forest  ===
Accuracy: 0.9451945194519452
F1 Score: 0.7132391418105704
Confusion Matrix:
 [[17539   759]
 [  337  1363]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.96      0.97     18298
           1       0.64      0.80      0.71      1700

    accuracy                           0.95     19998
   macro avg       0.81      0.88      0.84     19998
weighted avg       0.95      0.95      0.95     19998


=== XGBoost ===
Accuracy: 0.9393439343934393
F1 Score: 0.70159901599016
Confusion Matrix:
 [[17359   939]
 [  274  1426]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.95      0.97     18298
           1       0.60      0.84      0.70      1700

    accuracy                           0.94     19998
   macro avg       0.79      0.89      0.83     19998
weighted avg       0.95      0.94      

In [26]:
# 1️⃣ SMOTEENN
# ===============================
from imblearn.combine import SMOTEENN
smote_enn = SMOTEENN(random_state=42)
X_res, y_res = smote_enn.fit_resample(X_train, y_train)
print("Distribusi setelah SMOTEENN:", Counter(y_res))


# 2️⃣ Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)
rf.fit(X_res, y_res)
y_pred_rf = rf.predict(X_test)

print("\n=== Random Forest  ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf, pos_label=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))

# 3️⃣ XGBoost
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb.fit(X_res, y_res)
y_pred_xgb = xgb.predict(X_test)

print("\n=== XGBoost ===")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb, pos_label=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))


Distribusi setelah SMOTEENN: Counter({1: 72277, 0: 63393})

=== Random Forest  ===
Accuracy: 0.9451945194519452
F1 Score: 0.7132391418105704
Confusion Matrix:
 [[17539   759]
 [  337  1363]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.96      0.97     18298
           1       0.64      0.80      0.71      1700

    accuracy                           0.95     19998
   macro avg       0.81      0.88      0.84     19998
weighted avg       0.95      0.95      0.95     19998


=== XGBoost ===
Accuracy: 0.9393439343934393
F1 Score: 0.70159901599016
Confusion Matrix:
 [[17359   939]
 [  274  1426]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.95      0.97     18298
           1       0.60      0.84      0.70      1700

    accuracy                           0.94     19998
   macro avg       0.79      0.89      0.83     19998
weighted avg       0.95      0.94      

In [27]:
from imblearn.over_sampling import ADASYN
# ADASYN
from imblearn.over_sampling import ADASYN
adasyn = ADASYN(random_state=42)
X_res, y_res = adasyn.fit_resample(X_train, y_train)
print("Distribusi setelah ADASYN:", Counter(y_res))

# 2️⃣ Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)
rf.fit(X_res, y_res)
y_pred_rf = rf.predict(X_test)

print("\n=== Random Forest  ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf, pos_label=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))

# 3️⃣ XGBoost
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb.fit(X_res, y_res)
y_pred_xgb = xgb.predict(X_test)

print("\n=== XGBoost ===")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb, pos_label=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))

Distribusi setelah ADASYN: Counter({0: 73188, 1: 72980})

=== Random Forest  ===
Accuracy: 0.9628462846284629
F1 Score: 0.7634511302133079
Confusion Matrix:
 [[18056   242]
 [  501  1199]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.99      0.98     18298
           1       0.83      0.71      0.76      1700

    accuracy                           0.96     19998
   macro avg       0.90      0.85      0.87     19998
weighted avg       0.96      0.96      0.96     19998


=== XGBoost ===
Accuracy: 0.9673467346734673
F1 Score: 0.7859718125204851
Confusion Matrix:
 [[18146   152]
 [  501  1199]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.99      0.98     18298
           1       0.89      0.71      0.79      1700

    accuracy                           0.97     19998
   macro avg       0.93      0.85      0.88     19998
weighted avg       0.97      0.97      

In [28]:
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(random_state=42)
X_res, y_res = rus.fit_resample(X_train, y_train)
print("Distribusi setelah RUS:", Counter(y_res))

# 2️⃣ Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)
rf.fit(X_res, y_res)
y_pred_rf = rf.predict(X_test)

print("\n=== Random Forest  ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf, pos_label=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))

# 3️⃣ XGBoost
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb.fit(X_res, y_res)
y_pred_xgb = xgb.predict(X_test)

print("\n=== XGBoost ===")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb, pos_label=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))

Distribusi setelah RUS: Counter({0: 6800, 1: 6800})

=== Random Forest  ===
Accuracy: 0.8983898389838983
F1 Score: 0.6029699101211411
Confusion Matrix:
 [[16423  1875]
 [  157  1543]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.90      0.94     18298
           1       0.45      0.91      0.60      1700

    accuracy                           0.90     19998
   macro avg       0.72      0.90      0.77     19998
weighted avg       0.94      0.90      0.91     19998


=== XGBoost ===
Accuracy: 0.8989398939893989
F1 Score: 0.6097702259123383
Confusion Matrix:
 [[16398  1900]
 [  121  1579]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.90      0.94     18298
           1       0.45      0.93      0.61      1700

    accuracy                           0.90     19998
   macro avg       0.72      0.91      0.78     19998
weighted avg       0.95      0.90      0.91 

In [29]:
# 1️⃣ Resampling dengan ClusterCentroids
# ===============================
from imblearn.under_sampling import ClusterCentroids
cc = ClusterCentroids(random_state=42)
X_res, y_res = cc.fit_resample(X_train, y_train)
print("Distribusi setelah ClusterCentroids:", Counter(y_res))

# 2️⃣ Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)
rf.fit(X_res, y_res)
y_pred_rf = rf.predict(X_test)

print("\n=== Random Forest  ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf, pos_label=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))

# 3️⃣ XGBoost
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb.fit(X_res, y_res)
y_pred_xgb = xgb.predict(X_test)

print("\n=== XGBoost ===")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb, pos_label=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))

Distribusi setelah ClusterCentroids: Counter({0: 6800, 1: 6800})

=== Random Forest  ===
Accuracy: 0.6603660366036603
F1 Score: 0.3333333333333333
Confusion Matrix:
 [[11508  6790]
 [    2  1698]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.63      0.77     18298
           1       0.20      1.00      0.33      1700

    accuracy                           0.66     19998
   macro avg       0.60      0.81      0.55     19998
weighted avg       0.93      0.66      0.73     19998


=== XGBoost ===
Accuracy: 0.6559155915591559
F1 Score: 0.33044662839350003
Confusion Matrix:
 [[11419  6879]
 [    2  1698]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.62      0.77     18298
           1       0.20      1.00      0.33      1700

    accuracy                           0.66     19998
   macro avg       0.60      0.81      0.55     19998
weighted avg       0.93      0

In [30]:
from imblearn.under_sampling import TomekLinks
# ===============================
# 1️⃣ Resampling dengan Tomek Links
# ===============================
tl = TomekLinks()
X_res, y_res = tl.fit_resample(X_train, y_train)
print("Distribusi setelah Tomek Links:", Counter(y_res))

# 2️⃣ Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)
rf.fit(X_res, y_res)
y_pred_rf = rf.predict(X_test)

print("\n=== Random Forest  ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf, pos_label=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))

# 3️⃣ XGBoost
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb.fit(X_res, y_res)
y_pred_xgb = xgb.predict(X_test)

print("\n=== XGBoost ===")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb, pos_label=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))

Distribusi setelah Tomek Links: Counter({0: 72119, 1: 6800})

=== Random Forest  ===
Accuracy: 0.9705970597059705
F1 Score: 0.7990430622009569
Confusion Matrix:
 [[18241    57]
 [  531  1169]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.98     18298
           1       0.95      0.69      0.80      1700

    accuracy                           0.97     19998
   macro avg       0.96      0.84      0.89     19998
weighted avg       0.97      0.97      0.97     19998


=== XGBoost ===
Accuracy: 0.9722972297229723
F1 Score: 0.8088336783988958
Confusion Matrix:
 [[18272    26]
 [  528  1172]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.99     18298
           1       0.98      0.69      0.81      1700

    accuracy                           0.97     19998
   macro avg       0.98      0.84      0.90     19998
weighted avg       0.97      0.97  

In [31]:
from imblearn.under_sampling import EditedNearestNeighbours
enn = EditedNearestNeighbours()
X_res, y_res = enn.fit_resample(X_train, y_train)
print("Distribusi setelah ENN:", Counter(y_res))

# 2️⃣ Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)
rf.fit(X_res, y_res)
y_pred_rf = rf.predict(X_test)

print("\n=== Random Forest  ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf, pos_label=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))

# 3️⃣ XGBoost
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb.fit(X_res, y_res)
y_pred_xgb = xgb.predict(X_test)

print("\n=== XGBoost ===")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb, pos_label=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))

Distribusi setelah ENN: Counter({0: 67964, 1: 6800})

=== Random Forest  ===
Accuracy: 0.9672967296729673
F1 Score: 0.790920716112532
Confusion Matrix:
 [[18107   191]
 [  463  1237]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.99      0.98     18298
           1       0.87      0.73      0.79      1700

    accuracy                           0.97     19998
   macro avg       0.92      0.86      0.89     19998
weighted avg       0.97      0.97      0.97     19998


=== XGBoost ===
Accuracy: 0.9671467146714672
F1 Score: 0.7910969793322734
Confusion Matrix:
 [[18097   201]
 [  456  1244]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.99      0.98     18298
           1       0.86      0.73      0.79      1700

    accuracy                           0.97     19998
   macro avg       0.92      0.86      0.89     19998
weighted avg       0.97      0.97      0.97 

In [61]:
# =========================
# 0️⃣ Library
# =========================
import pandas as pd
import pickle
from collections import Counter
from xgboost import XGBClassifier
from imblearn.under_sampling import TomekLinks
import warnings
warnings.filterwarnings('ignore')

# =========================
# 1️⃣ Load dataset training
# =========================
data = pd.read_csv("diabetes_dataset.csv")  # dataset asli dengan target 'diabetes'
X = data.drop("diabetes", axis=1)
y = data["diabetes"]

# =========================
# 2️⃣ One-Hot Encoding kolom kategorikal
# =========================
categorical_cols = ["gender", "smoking_history", "location"]  # semua kolom kategori
for col in categorical_cols:
    X[col] = X[col].astype(str)

X_encoded = pd.get_dummies(X, columns=categorical_cols)

# Simpan daftar kolom training
model_columns = X_encoded.columns.tolist()
pickle.dump(model_columns, open("model_columns.pkl", "wb"))
print("✅ model_columns.pkl berhasil dibuat")

# =========================
# 3️⃣ Balancing dengan TomekLinks
# =========================
tl = TomekLinks()
X_res, y_res = tl.fit_resample(X_encoded, y)
print("Distribusi target setelah TomekLinks:", Counter(y_res))

# =========================
# 4️⃣ Latih XGBoost
# =========================
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb.fit(X_res, y_res)
print("✅ Model XGBoost dengan TomekLinks sudah dilatih")

# Simpan model
pickle.dump(xgb, open("model_xgb_tomek.pkl", "wb"))
print("✅ model_xgb_tomek.pkl berhasil dibuat")

# =========================
# 5️⃣ Prediksi data baru
# =========================
data_baru = pd.read_csv("diabetes_dataset.csv")  # dataset baru tanpa target

# One-Hot Encoding kolom kategorikal
for col in categorical_cols:
    data_baru[col] = data_baru[col].astype(str)
data_baru_encoded = pd.get_dummies(data_baru, columns=categorical_cols)

# Tambahkan kolom yang hilang agar sesuai training
for c in model_columns:
    if c not in data_baru_encoded.columns:
        data_baru_encoded[c] = 0

# Urutkan kolom sesuai training
data_baru_encoded = data_baru_encoded[model_columns]

# Prediksi
model_xgb = pickle.load(open("model_xgb_tomek.pkl", "rb"))
data_baru["prediksi_diabetes"] = model_xgb.predict(data_baru_encoded)

# =========================
# 6️⃣ Simpan hasil prediksi
# =========================
data_baru.to_csv("hasil_prediksi_xgb_tomek.csv", index=False)
print("✅ Hasil prediksi tersimpan di 'hasil_prediksi_xgb_tomek.csv'")

✅ model_columns.pkl berhasil dibuat
Distribusi target setelah TomekLinks: Counter({0: 90104, 1: 8500})
✅ Model XGBoost dengan TomekLinks sudah dilatih
✅ model_xgb_tomek.pkl berhasil dibuat
✅ Hasil prediksi tersimpan di 'hasil_prediksi_xgb_tomek.csv'
